In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

DATASET_PATH = os.path.join(PROJECT_PATH, "dataset")

TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
VAL_CSV   = os.path.join(DATASET_PATH, "val.csv")
TEST_CSV  = os.path.join(DATASET_PATH, "test.csv")

EXTRACTED_PATH = os.path.join(PROJECT_PATH, "extracted_frames")

os.makedirs(EXTRACTED_PATH, exist_ok=True)

print("Ready")

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(train_df.head())

In [ ]:
sample = train_df.iloc[0]

video_path = os.path.join(
    DATASET_PATH,
    sample["folder"],
    sample["filename"]
)

print(video_path)
print(os.path.exists(video_path))

In [ ]:
cap = cv2.VideoCapture(video_path)

print("Opened:", cap.isOpened())

length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Frames:", length)

cap.release()

In [ ]:
import cv2
import numpy as np

def calculate_motion_score(prev_frame, curr_frame):
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

    diff = cv2.absdiff(prev_gray, curr_gray)

    return np.mean(diff)

In [ ]:
import os
import cv2
import numpy as np

# Output folder
test_output = os.path.join(PROJECT_PATH, "informative_frames_test")
os.makedirs(test_output, exist_ok=True)

# Clear previous images
for f in os.listdir(test_output):
    os.remove(os.path.join(test_output, f))

cap = cv2.VideoCapture(video_path)

frames = []
motion_scores = []

ret, prev_frame = cap.read()

if not ret:
    print("Unable to read video")

frame_index = 0

while True:
    ret, curr_frame = cap.read()

    if not ret:
        break

    score = calculate_motion_score(prev_frame, curr_frame)

    frames.append((frame_index, curr_frame.copy()))
    motion_scores.append((frame_index, score))

    prev_frame = curr_frame
    frame_index += 1

cap.release()

# -------------------------------
# Select Top-5 Informative Frames
# with Minimum Gap
# -------------------------------

sorted_scores = sorted(motion_scores, key=lambda x: x[1], reverse=True)

selected = []
MIN_GAP = 20

for idx, score in sorted_scores:

    if all(abs(idx - prev_idx) >= MIN_GAP for prev_idx, _ in selected):
        selected.append((idx, score))

    if len(selected) == 5:
        break

print("\nTop-5 Informative Frames")

for idx, score in selected:
    print(f"Frame {idx}  Motion Score = {score:.2f}")

# Save Frames

for rank, (idx, score) in enumerate(selected):

    frame = frames[idx][1]

    cv2.imwrite(
        os.path.join(
            test_output,
            f"frame_{rank+1}_{idx}.jpg"
        ),
        frame
    )

print("\n✅ Top-5 Informative Frames Saved Successfully")

In [ ]:
print(os.listdir(test_output))

In [ ]:
import matplotlib.pyplot as plt

files = sorted(os.listdir(test_output))

plt.figure(figsize=(15,5))

for i, file in enumerate(files):
    img = cv2.imread(os.path.join(test_output, file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.title(file)
    plt.axis("off")

plt.show()

In [ ]:
import os
import cv2
from tqdm import tqdm

OUTPUT_PATH = os.path.join(PROJECT_PATH, "informative_frames")

os.makedirs(OUTPUT_PATH, exist_ok=True)

def extract_informative_frames(video_path, save_folder):

    os.makedirs(save_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frames = []
    motion_scores = []

    ret, prev_frame = cap.read()

    if not ret:
        cap.release()
        return

    frame_index = 0

    while True:

        ret, curr_frame = cap.read()

        if not ret:
            break

        score = calculate_motion_score(prev_frame, curr_frame)

        frames.append((frame_index, curr_frame.copy()))
        motion_scores.append((frame_index, score))

        prev_frame = curr_frame
        frame_index += 1

    cap.release()

    sorted_scores = sorted(motion_scores,
                           key=lambda x: x[1],
                           reverse=True)

    selected = []

    MIN_GAP = 20

    for idx, score in sorted_scores:

        if all(abs(idx-p)>=MIN_GAP for p,_ in selected):

            selected.append((idx,score))

        if len(selected)==5:
            break

    for rank,(idx,score) in enumerate(selected):

        frame = frames[idx][1]

        cv2.imwrite(
            os.path.join(
                save_folder,
                f"frame_{rank+1}.jpg"
            ),
            frame
        )

In [ ]:
all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("Total Videos :", len(all_df))

for _, row in tqdm(all_df.iterrows(),
                   total=len(all_df)):

    video_path = os.path.join(
        DATASET_PATH,
        row["folder"],
        row["filename"]
    )

    video_name = os.path.splitext(row["filename"])[0]

    save_folder = os.path.join(
        OUTPUT_PATH,
        row["folder"],
        video_name
    )

    extract_informative_frames(
        video_path,
        save_folder
    )

print("\n✅ Informative Frame Extraction Completed")

In [ ]:
print(os.listdir(OUTPUT_PATH))

In [ ]:
print(len(os.listdir(os.path.join(OUTPUT_PATH, "original"))))
print(len(os.listdir(os.path.join(OUTPUT_PATH, "Deepfakes"))))

In [ ]:
sample_video = os.listdir(
    os.path.join(OUTPUT_PATH, "original")
)[0]

sample_path = os.path.join(
    OUTPUT_PATH,
    "original",
    sample_video
)

print(os.listdir(sample_path))